In [1]:
# ЗАДАЧА 1.1: Импортируйте все необходимые библиотеки
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import zipfile
import io
import urllib.request
from datasets import load_dataset

from sklearn.linear_model import LogisticRegression ,  Ridge
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler , LabelEncoder
from sklearn.ensemble import RandomForestRegressor ,VotingRegressor
from sklearn.metrics import mean_squared_error ,r2_score , mean_absolute_error

warnings.filterwarnings('ignore')
# Ваш код здесь


C:\Users\Legion5\PycharmProjects\ML_from_scratch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASET_URL = 'https://huggingface.co/datasets/raimbekovm/astana-real-estate/resolve/main/data/astana_apartments.csv'

def load_dataset(url: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(url,encoding='utf-8')
        return df
    except UnicodeDecodeError as e:
        raise RuntimeError("Enode error while reading dataset ") from e
    except Exception as e:
        raise RuntimeError("Failed to load_dataset ") from e


def get_information(df : pd.DataFrame) :
    display(df.head())
    print('=='*50)
    print(f"Shape of the data : {df.shape}")
    print('='*50)
    df.info()
    print('--'*50)
    display(df.describe())

df = load_dataset(DATASET_URL)
get_information(df)


,id,listing_id,url,source,residential_complex_id,residential_complex_name,rooms,area,living_area,kitchen_area,...,internet,documents,offer_type,address,district,latitude,longitude,parsed_at,updated_at,is_active
0,1,691048362,https://krisha.kz/a/show/691048362?srchid=019b...,krisha.kz,NaN,Respublika,2,72.50,NaN,NaN,...,NaN,NaN,NaN,Е-36 5,Нура р-н,51.128205,71.385412,2026-01-03T20:50:46.421397,NaN,1
1,2,1004599137,https://krisha.kz/a/show/1004599137?srchid=019...,krisha.kz,NaN,Family Town,3,89.30,NaN,11.80,...,NaN,NaN,NaN,Е652 6,Есильский р-н,51.059035,71.420952,2026-01-03T20:50:52.735945,NaN,1
2,3,761512600,https://krisha.kz/a/show/761512600?srchid=019b...,krisha.kz,NaN,Свечки,2,72.10,NaN,NaN,...,NaN,NaN,NaN,Богенбай батыра 30,Сарыарка р-н,51.189164,71.356273,2026-01-03T20:51:01.431439,NaN,1
3,4,1007029481,https://krisha.kz/a/show/1007029481?srchid=019...,krisha.kz,NaN,Sezim Qala.Baqyt,4,106.77,NaN,11.13,...,NaN,NaN,NaN,Туран 55/12,Нура р-н,51.114484,71.393405,2026-01-03T20:51:09.278359,NaN,1
4,5,697845378,https://krisha.kz/a/show/697845378?srchid=019b...,krisha.kz,NaN,Boston Comfort House,2,39.00,NaN,NaN,...,NaN,NaN,NaN,"мкр Уркер, Кургальжинское шоссе 108",Нура р-н,51.121439,71.283095,2026-01-03T20:51:19.144336,NaN,1


Shape of the data : (18388, 37)
<class 'pandas.DataFrame'>
RangeIndex: 18388 entries, 0 to 18387
Data columns (total 37 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id                        18388 non-null  int64  
 1   listing_id                18388 non-null  int64  
 2   url                       18388 non-null  str    
 3   source                    18388 non-null  str    
 4   residential_complex_id    0 non-null      float64
 5   residential_complex_name  14495 non-null  str    
 6   rooms                     18388 non-null  int64  
 7   area                      18388 non-null  float64
 8   living_area               82 non-null     float64
 9   kitchen_area              7621 non-null   float64
 10  floor                     17417 non-null  float64
 11  total_floors              17417 non-null  float64
 12  price_usd                 18388 non-null  int64  
 13  price_local               18388 non-null

,id,listing_id,residential_complex_id,rooms,area,living_area,kitchen_area,floor,total_floors,price_usd,...,year_built,building_series,ceiling_height,heating,internet,documents,offer_type,latitude,longitude,is_active
count,18388.000000,1.838800e+04,0.0,18388.000000,18388.000000,82.000000,7621.000000,17417.000000,17417.000000,1.838800e+04,...,18388.000000,0.0,15189.000000,0.0,0.0,0.0,0.0,18388.000000,18388.000000,18388.0
mean,9194.500000,9.725041e+08,NaN,2.317762,73.047921,71.270122,13.122130,6.767526,11.561750,9.395154e+04,...,2016.711225,NaN,2.896558,NaN,NaN,NaN,NaN,51.124212,71.426141,1.0
std,5308.302711,1.005727e+08,NaN,1.041853,41.691703,43.650309,10.682632,4.329900,5.109851,8.081485e+04,...,11.192800,NaN,0.610195,NaN,NaN,NaN,NaN,0.204731,0.334562,0.0
min,1.000000,1.940254e+07,NaN,1.000000,10.800000,18.800000,1.200000,1.000000,2.000000,7.840000e+03,...,1950.000000,NaN,2.000000,NaN,NaN,NaN,NaN,42.346735,51.905115,1.0
25%,4597.750000,1.005556e+09,NaN,2.000000,43.700000,41.625000,10.000000,3.000000,9.000000,5.010700e+04,...,2013.000000,NaN,2.700000,NaN,NaN,NaN,NaN,51.111286,71.391065,1.0
50%,9194.500000,1.007650e+09,NaN,2.000000,63.000000,60.000000,12.000000,6.000000,10.000000,7.073900e+04,...,2020.000000,NaN,3.000000,NaN,NaN,NaN,NaN,51.127912,71.426501,1.0
75%,13791.250000,1.008192e+09,NaN,3.000000,90.000000,86.000000,15.000000,9.000000,14.000000,1.080740e+05,...,2024.000000,NaN,3.000000,NaN,NaN,NaN,NaN,51.148870,71.468840,1.0
max,18388.000000,1.008408e+09,NaN,13.000000,556.400000,285.200000,707.000000,41.000000,43.000000,2.947475e+06,...,2028.000000,NaN,28.000000,NaN,NaN,NaN,NaN,55.231707,81.102141,1.0


In [4]:
def get_null_summary(df: pd.DataFrame) -> pd.DataFrame:
    null_count = df.isnull().sum()
    null_percent = (null_count/len(df)) * 100


    return (
        pd.DataFrame({
            "nulls": null_count,
            "percent": null_percent
        }).sort_values("percent",ascending=False)
    )

def get_collumn_data_type(df: pd.DataFrame):
    return (
        df.dtypes.rename("dtype").to_frame()
    )


print(get_null_summary(df))
print("=="*50)
get_collumn_data_type(df)

                          nulls     percent
residential_complex_id    18388  100.000000
internet                  18388  100.000000
documents                 18388  100.000000
heating                   18388  100.000000
building_series           18388  100.000000
offer_type                18388  100.000000
living_area               18306   99.554057
updated_at                14838   80.693931
condition                 10887   59.207092
kitchen_area              10767   58.554492
floor_type                10501   57.107896
security                  10143   55.160975
furniture                  9323   50.701544
balcony                    8981   48.841636
bathroom                   7164   38.960191
parking                    5737   31.199695
residential_complex_name   3893   21.171416
ceiling_height             3199   17.397216
house_type                 1117    6.074614
total_floors                971    5.280618
floor                       971    5.280618
district                    778 

,dtype
id,int64
listing_id,int64
url,str
source,str
residential_complex_id,float64
residential_complex_name,str
rooms,int64
area,float64
living_area,float64
kitchen_area,float64
